# Run CESM-SMYLE benchmark preprocessing

This notebook is a lightweight driver for `run_process_cesm_smyle_benchmark.py`.

It passes the required settings to the preprocessing module and calls `process_one(...)` for selected fields, initialization months, years, ensemble members, and output frequencies.

Typical use cases:

- monthly benchmark for Niño3.4 SST skill: `fields = ["TS"]`, `init_months = [5, 11]`, `freqs = ["mon"]`
- seasonal benchmark for skill maps: `fields = ["TREFHT", "PRECT", "PSL"]`, `init_months = [5, 11]`, `freqs = ["seas"]`
- both monthly and seasonal benchmark files: `freqs = ["mon", "seas"]`

Generated benchmark files are written under the clean model-first diagnostic layout:

- `/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/<FIELD>/`
- verification figures, when enabled, go to `/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/CESM-SMYLE/verify/`


In [17]:
# Basic imports
import os
import sys
import time
import traceback
import warnings
from pathlib import Path

import numpy as np
import xarray as xr

# Optional: Dask for parallel preprocessing
from dask.distributed import Client, LocalCluster
import dask

## 1. User settings

Edit this block for the dataset/variable you want to process.


In [18]:
# Path to the preprocessing script.
# If the .py file is in the same directory as this notebook, keep this as is.
script_path = Path("/global/homes/z/zhan391/code/ESP-Lab/scripts/run_process_cesm_smyle_benchmark.py")

# Input/output paths
data_dir = "/global/cfs/cdirs/e3sm/S2S2D/CESM-SMYLE"
outdir = "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE"
figdir = "/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag"

# Variables and initialization months
# For Niño3.4 SST monthly skill, use TS.
fields = ["TS", "TREFHT", "PRECT", "PSL"]

# Usually May and November for the E3SM S2D comparison.
init_months = [2, 5, 8, 11]

# Choose benchmark frequencies to write.
#   "mon"  = monthly benchmark, needed for monthly Niño3.4 bottom panel
#   "seas" = seasonal benchmark, needed for seasonal skill maps/top panels
freqs = ["mon", "seas"]

# Years, members, and lead months
year_start = 1980
year_end = 2018
nens = 20
nlead = 24

# Processing behavior
force = False
dry_run = False
require_all_members = True
verify_coverage = True
open_parallel = False
run_verify = False

# Dask settings
use_dask = False
workers = 32
threads_per_worker = 1

print("script_path:", script_path)
print("fields:", fields)
print("init_months:", init_months)
print("freqs:", freqs)

script_path: /global/homes/z/zhan391/code/ESP-Lab/scripts/run_process_cesm_smyle_benchmark.py
fields: ['TS', 'TREFHT', 'PRECT', 'PSL']
init_months: [2, 5, 8, 11]
freqs: ['mon', 'seas']


## 2. Import the preprocessing module

This imports the `.py` script as a module, so we can call `process_one(...)` directly from the notebook.


In [19]:
import importlib.util

if not script_path.exists():
    raise FileNotFoundError(
        f"Cannot find preprocessing script: {script_path}\n"
        "Put run_process_cesm_smyle_benchmark.py in the same directory "
        "as this notebook, or update script_path above."
    )

spec = importlib.util.spec_from_file_location("run_process_cesm_smyle_benchmark", script_path)
prep = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prep)

if outdir is None:
    outdir = prep.OUTDIR_DEFAULT
if figdir is None:
    figdir = prep.FIGDIR_DEFAULT

print("Imported module from:", script_path)
print("Benchmark outdir:", outdir)

Imported module from: /global/homes/z/zhan391/code/ESP-Lab/scripts/run_process_cesm_smyle_benchmark.py
Benchmark outdir: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE


## 3. Start Dask client

The preprocessing module uses Dask/xarray internally. Starting a local Dask client here helps parallelize loading, seasonal aggregation, and NetCDF writing.


In [20]:
client = None
cluster = None

if use_dask:
    dask.config.set({"array.slicing.split_large_chunks": True})
    cluster = LocalCluster(
        n_workers=workers,
        threads_per_worker=threads_per_worker,
    )
    client = Client(cluster)
    print("Dask dashboard:", client.dashboard_link)
else:
    print("Running without an explicit Dask distributed client.")

Running without an explicit Dask distributed client.


## 4. Build processing list

This mirrors the CLI logic from the `.py` script but keeps all settings visible in the notebook.


In [21]:
years = list(range(year_start, year_end + 1))
members = [f"EN{i:02d}" for i in range(1, nens + 1)]
combos = [(field, init_month) for field in fields for init_month in init_months]

print("=" * 70)
print("CESM-SMYLE benchmark preprocessing from notebook")
print("=" * 70)
print(f"data_dir      : {data_dir}")
print(f"outdir        : {outdir}")
print(f"figdir        : {figdir}")
print(f"fields        : {fields}")
print(f"init_months   : {init_months}")
print(f"freqs         : {freqs}")
print(f"years         : {years[0]}–{years[-1]}  ({len(years)} years)")
print(f"members       : EN01–EN{nens:02d}  ({nens} members)")
print(f"nlead         : {nlead} months")
print(f"combinations  : {len(combos)}")
print(f"force         : {force}")
print(f"dry_run       : {dry_run}")
print(f"open_parallel : {open_parallel}")
print(f"run_verify    : {run_verify}")
print("=" * 70)

CESM-SMYLE benchmark preprocessing from notebook
data_dir      : /global/cfs/cdirs/e3sm/S2S2D/CESM-SMYLE
outdir        : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE
figdir        : /global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag
fields        : ['TS', 'TREFHT', 'PRECT', 'PSL']
init_months   : [2, 5, 8, 11]
freqs         : ['mon', 'seas']
years         : 1980–2018  (39 years)
members       : EN01–EN20  (20 members)
nlead         : 24 months
combinations  : 16
force         : False
dry_run       : False
open_parallel : False
run_verify    : False


## 5. Run preprocessing

This calls:

```python
prep.process_one(...)
```

for each `(field, init_month)` combination.


In [22]:
%%time

total_t0 = time.perf_counter()
counters = {
    "ok": 0,
    "skipped": 0,
    "dry_run": 0,
    "no_data": 0,
    "failed": 0,
}

results = []

try:
    for field, init_month in combos:
        print("\n" + "-" * 70)
        print(f"Processing field={field}, init_month={init_month:02d}, freqs={freqs}")
        print("-" * 70)

        try:
            status = prep.process_one(
                field=field,
                init_month=init_month,
                data_dir=data_dir,
                outdir=outdir,
                years=years,
                members=members,
                nlead=nlead,
                require_all_members=require_all_members,
                verify_coverage=verify_coverage,
                open_parallel=open_parallel,
                force=force,
                dry_run=dry_run,
                run_verify=run_verify,
                freqs=freqs,
                figdir=figdir,
            )
            counters[status] = counters.get(status, 0) + 1

        except KeyboardInterrupt:
            print("\nInterrupted by user.")
            break

        except Exception as exc:
            traceback.print_exc()
            warnings.warn(f"FAILED: field={field}, init_month={init_month}: {exc}")
            status = "failed"
            counters["failed"] += 1

        results.append(
            {
                "field": field,
                "init_month": init_month,
                "freqs": ",".join(freqs),
                "status": status,
            }
        )

finally:
    elapsed = time.perf_counter() - total_t0
    print("\n" + "=" * 70)
    print(f"Finished in {elapsed:.1f}s")
    print(f"Written:   {counters['ok']}")
    print(f"Skipped:   {counters['skipped']}")
    print(f"Dry-run:   {counters['dry_run']}")
    print(f"No data:   {counters['no_data']}")
    print(f"Failed:    {counters['failed']}")
    print("=" * 70)


----------------------------------------------------------------------
Processing field=TS, init_month=02, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=TS, init_month=05, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=TS, init_month=08, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=TS, init_month=11, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=TREFHT, init_month=02, freqs=['mon', 'seas']
--------------------------------------------------------------

## 6. Review outputs

This cell lists expected output files using the module's benchmark filename convention.


In [23]:
expected_files = []

for field in fields:
    for init_month in init_months:
        for freq in freqs:
            fname = prep.benchmark_filename(
                field,
                init_month,
                nens=nens,
                nlead=nlead,
                freq=freq,
            )
            expected_files.append(Path(outdir) / "leadtime_acc" / "inputs" / field / fname)

for path in expected_files:
    if path.exists():
        size_gb = path.stat().st_size / 1024**3
        print(f"[OK]      {path}  ({size_gb:.2f} GB)")
    else:
        print(f"[MISSING] {path}")

[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/TS/BSMYLE02_TS_N20_M24_mon.nc  (2.15 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/TS/BSMYLE02_TS_N20_M24_seas.nc  (0.63 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/TS/BSMYLE05_TS_N20_M24_mon.nc  (2.15 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/TS/BSMYLE05_TS_N20_M24_seas.nc  (0.63 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/TS/BSMYLE08_TS_N20_M24_mon.nc  (2.15 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/TS/BSMYLE08_TS_N20_M24_seas.nc  (0.63 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/TS/BSMYLE11_TS_N20_M24_mon.nc  (2.15 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/TS/BSMYLE11_TS_N20_M24_seas.nc  (0.63 GB)
[OK]      /global/cfs/cdirs/e3sm/S2S

## 7. Close Dask client

Run this after preprocessing is complete if `use_dask=True`.


In [24]:
if client is not None:
    client.close()
    print("Closed Dask client.")

if cluster is not None:
    cluster.close()
    print("Closed Dask cluster.")
